In [1]:
import pandas as pd
import numpy as np


In [2]:
file_path = r"C:\Users\husna\OneDrive\Documents\bsc-4th-ca-2-data-visualisation-and-communication-Husnain-Yaqoob\Online Retail (1).xlsx"
df = pd.read_excel(file_path)
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [13]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


# now that i have loaded the dataset i have  checked  the structure of the dataset to understand it's types,missing value patterns and the overall schema

## Missing values Check will be done from here on

In [5]:
df.isnull().sum()


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

# from the above output we can see that labels Description and CustomerID have missing values rest of the dataset is completely fine

In [6]:
df = df.dropna(subset=["Description"])
df.isnull().sum()


InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133626
Country             0
dtype: int64

Rows with missing product descriptions were removed because product analysis requires a valid Description. These rows were few and removing them improves data quality without affecting overall results.

In [7]:
print("Duplicate rows found:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates removed. New dataset shape:", df.shape)


Duplicate rows found: 5268
Duplicates removed. New dataset shape: (535187, 8)


 A total of 5,268 duplicate rows were found in the dataset. Duplicate transaction lines can artificially inflate sales figures and distort analysis, so they were removed. After dropping duplicates, the cleaned dataset now contains 535,187 unique records, ensuring that each transaction is counted only once.

In [8]:
# Remove cancelled invoices (InvoiceNo starting with 'C')
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Remove negative quantities i-e returns
df = df[df["Quantity"] > 0]

# Remove zero or negative prices i-e wrong prices
df = df[df["UnitPrice"] > 0]

df.shape


(524878, 8)

Invoices beginning with “C” represent cancelled or refunded orders and are not  completed sales were removed.
Negative quantities indicate returns, and zero or negative prices are invalid entries, so these were also filtered out.
After removing these records, the dataset now contains 524,878 valid sales transactions, providing a clean foundation for accurate analysis.

# conversion required for cleaning

In [9]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].head()


0   2010-12-01 08:26:00
1   2010-12-01 08:26:00
2   2010-12-01 08:26:00
3   2010-12-01 08:26:00
4   2010-12-01 08:26:00
Name: InvoiceDate, dtype: datetime64[ns]

InvoiceDate was converted to a proper datetime format so that time-based analysis (such as daily or monthly sales trends) can be performed accurately. This also enables date filtering later in the interactive dashboard.


# now as per assignment requirement we need to add atleast one variable that could be best suited to this dataset i am going with totalprice option


In [15]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df[["Quantity", "UnitPrice", "TotalPrice"]]


,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
...,...,...,...
541904,12,0.85,10.20
541905,6,2.10,12.60
541906,4,4.15,16.60
541907,4,4.15,16.60


A new column, TotalPrice, was generated by combining Quantity and UnitPrice.  This shows how much money each transaction line brings in.  It is necessary for creating the interactive dashboard and for assessing country revenue, top goods, and overall sales.